# `get_box_corners()`

The geometry function `nematics3d.get_box_corners()` returns the eight corner coordinates of an axis-aligned three-dimensional box whose minimum corner is the origin. The three inputs specify the box lengths along the $x$, $y$, and $z$ axes.

The function is useful when a box needs to be represented explicitly by its vertices, for example before applying a coordinate transform or constructing box edges and faces. The returned corner order is fixed because downstream geometry code may use the corner indices to define topology.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example

A box with side lengths 2, 3, and 4 spans from $(0,0,0)$ to $(2,3,4)$. `get_box_corners()` returns all eight combinations of the minimum and maximum coordinate along each axis.


In [2]:
corners = n3d.get_box_corners(2, 3, 4)
print(corners)

[[0. 0. 0.]
 [2. 0. 0.]
 [0. 3. 0.]
 [0. 0. 4.]
 [2. 3. 0.]
 [2. 0. 4.]
 [0. 3. 4.]
 [2. 3. 4.]]


## Inputs and outputs

The public signature is:

```python
get_box_corners(length_x, length_y, length_z)
```

`length_x`, `length_y`, and `length_z` must each be a finite, non-negative real scalar. Python and `NumPy` real-number scalars are accepted. Negative values, booleans, complex numbers, `NaN`, and infinite values are rejected.

The return value is a floating-point `NumPy` array with shape `(8, 3)`. Each row is one corner coordinate in $(x,y,z)$ order.

The corner indices are fixed as follows:

| Index | Coordinate |
| ---: | --- |
| 0 | $(0,0,0)$ |
| 1 | $(L_x,0,0)$ |
| 2 | $(0,L_y,0)$ |
| 3 | $(0,0,L_z)$ |
| 4 | $(L_x,L_y,0)$ |
| 5 | $(L_x,0,L_z)$ |
| 6 | $(0,L_y,L_z)$ |
| 7 | $(L_x,L_y,L_z)$ |

This ordering is part of the function contract. Code that defines box edges or faces by corner index can therefore use the returned array directly.


## Examples


### Translating a box

`get_box_corners()` always places the minimum corner at the origin. A translated axis-aligned box is obtained by adding its desired minimum-corner coordinate to every returned vertex.


In [3]:
origin = np.array([10.0, -2.0, 5.0])
translated = n3d.get_box_corners(2, 3, 4) + origin

print("minimum corner:", translated[0])
print("maximum corner:", translated[7])

minimum corner: [10. -2.  5.]
maximum corner: [12.  1.  9.]


### Zero length and degenerate boxes

A zero length is valid. In that case several of the eight corner rows coincide because the box collapses along that axis. This is useful for geometry derived from a lattice with only one point along a dimension.


In [4]:
flat_corners = n3d.get_box_corners(2, 0, 4)
print(flat_corners)

[[0. 0. 0.]
 [2. 0. 0.]
 [0. 0. 0.]
 [0. 0. 4.]
 [2. 0. 0.]
 [2. 0. 4.]
 [0. 0. 4.]
 [2. 0. 4.]]


## Details

The function constructs exactly eight vertices and does not allocate a coordinate mesh. The explicit ordering is intentional: it makes the relationship between corner indices and box topology transparent and avoids changing that ordering as an incidental consequence of another array-generation routine.

The returned coordinates describe only an axis-aligned box anchored at the origin. Translation, rotation, or a more general linear transform should be applied afterward when world-space geometry is required.


## Possible issues

- Do not assume the rows are lexicographically sorted. Use the documented corner indices when topology depends on ordering.
- A zero length is accepted and intentionally produces repeated coordinates. If a later operation requires a three-dimensional box with nonzero volume, validate that requirement at that later operation.
- The function does not accept a three-element length vector as one positional argument; pass the three axis lengths separately, for example `get_box_corners(*lengths)`.
